# Notebook 07: Attribute Inference

## Why This Matters
A credit model trained to predict default risk doesn't use race as a feature — it would be illegal. But it uses zip code, employment history, and credit utilization. An attacker with black-box API access can query the model and infer race anyway, because those correlated features encode it. This is attribute inference: recovering sensitive attributes not in the model's input, purely from model outputs.

This isn't a theoretical threat. FTC complaints against algorithmic systems routinely involve attribute inference through proxies. European GDPR and US fair lending regulations both have implicit exposure to this attack surface. Understanding it is prerequisite to building fair and private ML systems.

## Structure
1. What is attribute inference and how it differs from membership inference
2. The correlation attack: inferring attributes through correlated features
3. Shadow model attribute inference (Fredrikson et al. style)
4. When attribute inference is a fairness problem vs a privacy problem
5. Measuring leakage: mutual information and attribute inference accuracy
6. Defenses: adversarial debiasing, feature removal, representation learning

## What You'll Learn
- Why removing a sensitive feature doesn't prevent its inference
- How to measure how much a model's outputs reveal about protected attributes
- The connection between attribute inference and algorithmic fairness
- Which defenses actually reduce leakage vs just obscure it

## Background

### How attribute inference differs from membership inference

Membership inference asks: "was this record in the training set?" — a binary question about a known data point.

Attribute inference asks: "what are the sensitive attributes of this data point?" — a prediction task about an unknown feature value. The attacker typically has a partial record (some features known, one sensitive feature unknown) and uses model queries to fill in the missing piece.

The two attacks are related but distinct. Membership inference exploits the generalization gap. Attribute inference exploits the correlation structure of the training data — specifically, the fact that model outputs on correlated features implicitly encode the sensitive attribute.

### The correlation problem: removing the feature doesn't help

The naive mitigation for attribute inference is feature removal: if race is sensitive, don't use it as a model input. This is legally required in many contexts and widely practiced. It doesn't work.

The reason: proxy variables. Race in the United States is highly correlated with zip code, school quality, incarceration history, and credit history. A model that uses any of these features has implicitly encoded race in its feature space. An attacker who knows the values of the correlated features can infer the sensitive attribute from model outputs — even without the sensitive feature being present.

This is sometimes called the "fairness through unawareness" failure. Removing a protected attribute from the model doesn't make the model unaware of it; the information is encoded in the remaining features.

### Fredrikson et al. (2015) and the medical context

The same Fredrikson et al. paper that introduced model inversion also described attribute inference in the pharmacogenetics context. They showed that knowing a patient's warfarin dosage (the model output) plus non-sensitive input features allowed inference of the patient's genomic markers — sensitive medical information not in the model input.

The attack is straightforward: build a classifier that maps (model output + partial input) → missing attribute. If the model output encodes information correlated with the missing attribute, this classifier performs above chance.

### The fairness-privacy connection

Attribute inference sits at the intersection of fairness and privacy, and the connection is deep:

- A model that **discriminates** on a protected attribute encodes that attribute in its outputs by definition — which makes attribute inference easy
- A model that is **fair** (equal error rates across groups) still leaks the attribute if it achieves this by using proxy features
- **Demographic parity** (equal positive prediction rates across groups) is one of the few fairness criteria that actually prevents attribute inference — because if the model's output distribution is identical across groups, outputs carry no information about group membership

Researchers Hamman, Nori & Dutta (2023) formalized this: privacy-preserving models and fair models are not the same, and the conditions under which a model satisfies both are restrictive.

In [ ]:
%pip install -q torch numpy matplotlib scikit-learn pandas

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Synthetic Dataset: Credit Risk with Protected Attribute

We simulate a credit lending scenario:
- **Target**: default risk (0/1)
- **Sensitive attribute**: group membership (A/B) — not in model input
- **Model inputs**: credit score, income, debt ratio, employment length — all correlated with group

This mirrors real-world credit models where race/ethnicity correlates with socioeconomic features.

In [ ]:
def generate_credit_dataset(n=5000, seed=42):
    """
    Synthetic credit dataset with a protected group attribute.
    Group B has systematically lower credit scores and income
    due to simulated structural disadvantage — not included in model input.
    """
    rng = np.random.RandomState(seed)
    n = int(n)

    # Protected attribute: group A (0) vs group B (1), 60/40 split
    group = rng.binomial(1, 0.4, n)

    # Features are correlated with group — this is the structural problem
    credit_score    = rng.normal(680 - 60 * group, 80)   # group B: lower by 60 pts
    income          = rng.normal(55000 - 15000 * group, 20000)
    debt_ratio      = rng.normal(0.35 + 0.15 * group, 0.1).clip(0.01, 0.99)
    employment_yrs  = rng.normal(6 - 2 * group, 3).clip(0, 30)

    # Default probability is a function of features (not group directly)
    log_odds = (
        -3.0
        + 0.005 * (700 - credit_score)
        + 0.00003 * (60000 - income)
        + 4.0 * debt_ratio
        - 0.1 * employment_yrs
        + rng.normal(0, 0.3, n)
    )
    default_prob = 1 / (1 + np.exp(-log_odds))
    default = rng.binomial(1, default_prob)

    df = pd.DataFrame({
        'credit_score':   credit_score,
        'income':         income,
        'debt_ratio':     debt_ratio,
        'employment_yrs': employment_yrs,
        'group':          group,        # protected — not used in model
        'default':        default,      # target
    })
    return df


df = generate_credit_dataset(n=5000)
print("Dataset summary:")
print(df.describe().round(2))
print()
print(f"Default rate — Group A: {df[df.group==0]['default'].mean():.3f}")
print(f"Default rate — Group B: {df[df.group==1]['default'].mean():.3f}")
print()
print("Feature correlations with protected group attribute:")
for col in ['credit_score', 'income', 'debt_ratio', 'employment_yrs']:
    corr = df[col].corr(df['group'])
    print(f"  {col:18}: r = {corr:.3f}")

## 2. Training a "Fair" Model (Without the Protected Attribute)

In [ ]:
from sklearn.model_selection import train_test_split

FEATURES = ['credit_score', 'income', 'debt_ratio', 'employment_yrs']
TARGET   = 'default'
SENSITIVE = 'group'

X = df[FEATURES].values
y = df[TARGET].values
g = df[SENSITIVE].values

X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(
    X, y, g, test_size=0.3, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Train a standard logistic regression (no group feature)
clf = LogisticRegression(max_iter=500)
clf.fit(X_train_s, y_train)

train_proba = clf.predict_proba(X_train_s)[:, 1]
test_proba  = clf.predict_proba(X_test_s)[:, 1]

print("Credit model performance (no protected attribute in input):")
print(f"  Test AUC:      {roc_auc_score(y_test, test_proba):.4f}")
print(f"  Test accuracy: {accuracy_score(y_test, (test_proba > 0.5).astype(int)):.4f}")
print()

# Show model outputs differ by group — evidence of implicit encoding
test_df = pd.DataFrame({'group': g_test, 'pred_proba': test_proba, 'actual': y_test})
print("Mean predicted default probability by group:")
print(test_df.groupby('group')['pred_proba'].describe().round(3))
print()
print("The model never saw 'group' — but its outputs differ by group.")
print("This difference is what the attacker exploits.")

## 3. The Attribute Inference Attack

The attack: train a meta-classifier to predict group membership from (model output, partial input). The attacker is assumed to know all features except the sensitive one, plus the model's output probability.

In [ ]:
def attribute_inference_attack(clf_target, X_train, X_test, g_train, g_test, scaler):
    """
    Train an attack classifier to infer the protected attribute from:
      - Model's output probability (what the API returns)
      - All non-sensitive input features (known to attacker)
    """
    # Build attack features: [model output, known features]
    X_train_s = scaler.transform(X_train)
    X_test_s  = scaler.transform(X_test)

    pred_train = clf_target.predict_proba(X_train_s)[:, 1].reshape(-1, 1)
    pred_test  = clf_target.predict_proba(X_test_s)[:, 1].reshape(-1, 1)

    # Attack features: model output + all input features (attacker has both)
    attack_train = np.hstack([pred_train, X_train_s])
    attack_test  = np.hstack([pred_test,  X_test_s])

    # Attack without model output (baseline: just using features)
    baseline_train = X_train_s
    baseline_test  = X_test_s

    # Train attack classifier
    attack_clf   = LogisticRegression(max_iter=500)
    baseline_clf = LogisticRegression(max_iter=500)

    attack_clf.fit(attack_train, g_train)
    baseline_clf.fit(baseline_train, g_train)

    attack_auc   = roc_auc_score(g_test, attack_clf.predict_proba(attack_test)[:, 1])
    baseline_auc = roc_auc_score(g_test, baseline_clf.predict_proba(baseline_test)[:, 1])

    return attack_auc, baseline_auc, attack_clf


attack_auc, baseline_auc, attack_clf = attribute_inference_attack(
    clf, X_train, X_test, g_train, g_test, scaler
)

print("Attribute Inference Attack Results:")
print(f"  Baseline AUC (features only, no model):    {baseline_auc:.4f}")
print(f"  Attack AUC (features + model output):      {attack_auc:.4f}")
print(f"  Leakage from model output:                 {attack_auc - baseline_auc:+.4f}")
print()
print("The model output adds information about the protected attribute")
print("beyond what the features alone reveal.")

In [ ]:
# Visualize: model output distributions by group
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

test_proba = clf.predict_proba(scaler.transform(X_test))[:, 1]

axes[0].hist(test_proba[g_test==0], bins=40, alpha=0.6, color='steelblue', label='Group A', density=True)
axes[0].hist(test_proba[g_test==1], bins=40, alpha=0.6, color='darkorange', label='Group B', density=True)
axes[0].set_xlabel('Predicted default probability')
axes[0].set_ylabel('Density')
axes[0].set_title('Model Output by Group\n(protected attribute never in input)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Mutual information proxy: how much does model output vary with group?
thresholds = np.linspace(0, 1, 50)
frac_group_b = [test_proba[g_test==1][test_proba[g_test==1] > t].shape[0] /
                max(test_proba[g_test==1].shape[0], 1) for t in thresholds]
frac_group_a = [test_proba[g_test==0][test_proba[g_test==0] > t].shape[0] /
                max(test_proba[g_test==0].shape[0], 1) for t in thresholds]

axes[1].plot(thresholds, frac_group_b, color='darkorange', label='Group B (P(output>t))')
axes[1].plot(thresholds, frac_group_a, color='steelblue', label='Group A (P(output>t))')
axes[1].set_xlabel('Threshold t')
axes[1].set_ylabel('P(output > t)')
axes[1].set_title('Empirical CDF by Group\nGap = exploitable leakage')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('attribute_inference.png', bbox_inches='tight')
plt.show()

## 4. Measuring Leakage: Mutual Information

The right theoretical measure of how much a model's output reveals about a sensitive attribute is **mutual information**: $I(\hat{y}; A)$ where $\hat{y}$ is the model output and $A$ is the sensitive attribute.

If $I(\hat{y}; A) = 0$, the model output is independent of the attribute — perfect privacy. In practice, mutual information is estimated from data using histogram-based or k-NN estimators.

In [ ]:
from sklearn.metrics import mutual_info_score
from sklearn.preprocessing import KBinsDiscretizer

def estimate_mutual_information(y_pred: np.ndarray, sensitive: np.ndarray, n_bins: int = 20) -> float:
    """Estimate I(model_output; sensitive_attribute) via histogram binning."""
    discretizer = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='uniform')
    y_binned = discretizer.fit_transform(y_pred.reshape(-1, 1)).ravel().astype(int)
    return mutual_info_score(sensitive, y_binned)


# Measure MI for different models
print("Mutual information I(model output; group) — measures attribute leakage")
print("0.0 = no leakage, higher = more group information encoded in output")
print()

# Standard model
test_proba_std = clf.predict_proba(scaler.transform(X_test))[:, 1]
mi_std = estimate_mutual_information(test_proba_std, g_test)
print(f"  Standard model:                   MI = {mi_std:.4f}")

# Model with the protected attribute explicitly included
X_with_group = np.hstack([X, g.reshape(-1, 1)])
Xg_tr, Xg_te, _, _ = train_test_split(X_with_group, y, test_size=0.3, random_state=42)
sc2 = StandardScaler(); Xg_tr_s = sc2.fit_transform(Xg_tr); Xg_te_s = sc2.transform(Xg_te)
clf_with_group = LogisticRegression(max_iter=500).fit(Xg_tr_s, y_train)
proba_with_group = clf_with_group.predict_proba(Xg_te_s)[:, 1]
mi_with_group = estimate_mutual_information(proba_with_group, g_test)
print(f"  Model WITH protected attribute:   MI = {mi_with_group:.4f}  (upper bound)")

# Random predictions (no leakage)
random_pred = np.random.uniform(0, 1, len(g_test))
mi_random = estimate_mutual_information(random_pred, g_test)
print(f"  Random predictions (baseline):    MI = {mi_random:.4f}  (lower bound)")
print()
print(f"  Leakage fraction: {(mi_std - mi_random) / (mi_with_group - mi_random + 1e-10):.3f}")
print("  → the model without group encodes this fraction of the group-explicit model's leakage")

## 5. Defenses: Adversarial Debiasing

Adversarial debiasing (Zhang et al., 2018) trains the model with a two-player adversarial game:
- **Predictor**: minimizes task loss (default prediction)
- **Adversary**: maximizes its ability to infer the protected attribute from the predictor's output

The predictor is trained to be good at prediction *while* making the adversary fail. At equilibrium, the predictor's outputs carry minimal information about the sensitive attribute — reducing both attribute inference risk and demographic disparity.

In [ ]:
class Predictor(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(),
            nn.Linear(64, 32),         nn.ReLU(),
            nn.Linear(32, 1),
        )
    def forward(self, x): return self.net(x)  # returns logit


class Adversary(nn.Module):
    """Tries to infer sensitive attribute from predictor output."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 16), nn.ReLU(),
            nn.Linear(16, 1),
        )
    def forward(self, pred_logit): return self.net(pred_logit)


def train_adversarial_debiasing(X_train, y_train, g_train, alpha=1.0, epochs=50):
    """
    Adversarial debiasing (Zhang et al., 2018).
    alpha: strength of adversarial penalty (higher = more debiasing, less task accuracy)
    """
    X_t  = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_t  = torch.tensor(y_train, dtype=torch.float32, device=device).unsqueeze(1)
    g_t  = torch.tensor(g_train, dtype=torch.float32, device=device).unsqueeze(1)

    predictor = Predictor(X_train.shape[1]).to(device)
    adversary  = Adversary().to(device)

    opt_p = torch.optim.Adam(predictor.parameters(), lr=1e-3)
    opt_a = torch.optim.Adam(adversary.parameters(), lr=1e-3)

    for epoch in range(epochs):
        predictor.train(); adversary.train()

        pred_logit = predictor(X_t)

        # Adversary step: maximize ability to infer group from predictor output
        adv_logit = adversary(pred_logit.detach())
        adv_loss  = F.binary_cross_entropy_with_logits(adv_logit, g_t)
        opt_a.zero_grad(); adv_loss.backward(); opt_a.step()

        # Predictor step: minimize task loss MINUS adversary's ability to infer group
        pred_logit  = predictor(X_t)
        adv_logit   = adversary(pred_logit)
        task_loss   = F.binary_cross_entropy_with_logits(pred_logit, y_t)
        adv_loss_p  = F.binary_cross_entropy_with_logits(adv_logit, g_t)
        # Subtract adversary loss: predictor is rewarded for confusing the adversary
        total_loss  = task_loss - alpha * adv_loss_p
        opt_p.zero_grad(); total_loss.backward(); opt_p.step()

    return predictor


print("Training adversarially debiased models at different alpha values...")
X_train_s_np = scaler.transform(X_train)
X_test_s_np  = scaler.transform(X_test)

print(f"{'alpha':>8} {'Task AUC':>12} {'Group MI':>12} {'Leakage reduction':>20}")
print("-" * 58)

# Baseline
print(f"{'Standard':>8} {roc_auc_score(y_test, test_proba_std):>12.4f} {mi_std:>12.4f} {'—':>20}")

for alpha in [0.5, 1.0, 2.0, 5.0]:
    deb_model = train_adversarial_debiasing(X_train_s_np, y_train, g_train, alpha=alpha, epochs=100)
    deb_model.eval()
    with torch.no_grad():
        X_te_t = torch.tensor(X_test_s_np, dtype=torch.float32, device=device)
        deb_proba = torch.sigmoid(deb_model(X_te_t)).cpu().numpy().ravel()
    task_auc = roc_auc_score(y_test, deb_proba)
    mi_deb   = estimate_mutual_information(deb_proba, g_test)
    reduction = (mi_std - mi_deb) / mi_std * 100
    print(f"{alpha:>8.1f} {task_auc:>12.4f} {mi_deb:>12.4f} {reduction:>19.1f}%")

## Summary

**Attribute inference in one line:**  
If a model's outputs differ across groups, those outputs encode the group — and an attacker who knows correlated features can infer which group a person belongs to from the model output alone.

**Key facts:**
- Removing a sensitive feature doesn't prevent attribute inference if proxy features remain
- Mutual information $I(\hat{y}; A)$ is the principled measure of leakage
- Demographic parity (equal output distributions across groups) is the only fairness criterion that eliminates attribute inference
- Adversarial debiasing explicitly penalizes the model for encoding group information
- Higher alpha → more debiasing → lower task accuracy: a fundamental tradeoff

**Defense priority:**
1. Measure leakage before deployment using MI or attribute inference AUC
2. If leakage is high and regulatory exposure is real: adversarial debiasing or representation-based fairness
3. Output perturbation reduces leakage at low cost (same as model inversion defense)

**Next:** Notebook 08 shifts to training-time attacks — backdoor/trojan attacks, where an adversary with write access to the training set implants a hidden trigger that causes targeted misclassification.